# **02 - Snapshot Features**

This notebook builds the matchup-level feature table used to train the XGBoost model. Starting from `tracking_df` in `01_ingest.ipynb`, it pairs each blocker with their assigned rusher frame by frame, derives geometry features like distance and closing speed, and reduces each matchup to a single row sampled at the snap, roughly one second in, and the final tracked frame, with a binary outcome label built from PFF's sack, hurry, hit, and beaten-by-defender flags. Play-level context (down, distance, formation, and coverage scheme) is joined in, with sparse categories collapsed into an `other` bucket. The result is saved to `snapshot_features.parquet` as the handoff to `03_xgboost.ipynb`.

In [ ]:
%run 01_ingest.ipynb

In [ ]:
blockers = tracking_df[tracking_df["pff_role"] == "Pass Block"].copy()

print(f"Blocker-frame rows: {len(blockers):,}")
print(f"Unique blocker matchups (game/play/nflId): " f"{blockers[['gameId','playId','nflId']].drop_duplicates().shape[0]:,}")

In [ ]:
rusher_cols = ["gameId", "playId", "frameId", "nflId", "x", "y", "s", "a", "o", "dir"]
rushers = tracking_df[rusher_cols].rename(columns={
    "nflId": "rusher_nflId",
    "x": "rusher_x", "y": "rusher_y",
    "s": "rusher_s", "a": "rusher_a",
    "o": "rusher_o", "dir": "rusher_dir",
})

In [ ]:
paired = blockers.merge(
    rushers,
    left_on = ["gameId", "playId", "frameId", "pff_nflIdBlockedPlayer"],
    right_on = ["gameId", "playId", "frameId", "rusher_nflId"],
    how = "inner",
)

print(f"Paired blocker-rusher frame rows: {len(paired):,}")

In [ ]:
player_bio = players[["nflId", "height", "weight", "officialPosition"]]

paired = paired.merge(
    player_bio.rename(columns={
        "height": "blocker_height", "weight": "blocker_weight",
        "officialPosition": "blocker_position"
    }),
    on="nflId", how = "left",
)

paired = paired.merge(
    player_bio.rename(columns={
        "nflId": "rusher_nflId", "height": "rusher_height",
        "weight": "rusher_weight", "officialPosition": "rusher_position"
    }),
    on = "rusher_nflId", how = "left",
)

In [ ]:
import numpy as np

paired["distance"] = np.sqrt(
    (paired["x"] - paired["rusher_x"])**2 + (paired["y"] - paired["rusher_y"])**2
)

paired = paired.sort_values(["gameId", "playId", "nflId", "frameId"])
paired["prev_distance"] = paired.groupby(["gameId", "playId", "nflId"])["distance"].shift(1)
paired["closing_speed"] = paired["prev_distance"] - paired["distance"]

paired["orientation_diff"] = (paired["o"] - paired["rusher_o"]).abs()
paired["weight_diff"] = paired["blocker_weight"] - paired["rusher_weight"]

In [ ]:
def get_snapshots(group: pd.DataFrame) -> pd.Series:
    group = group.sort_values("frameId")
    snap_frame = group.iloc[0]
    mid_idx = min(9, len(group) - 1)
    mid_frame = group.iloc[mid_idx]
    final_frame = group.iloc[-1]

    return pd.Series({
        "dist_snap": snap_frame["distance"],
        "dist_1s": mid_frame["distance"],
        "dist_final": final_frame["distance"],
        "closing_speed_1s": mid_frame["closing_speed"],
        "closing_speed_final": final_frame["closing_speed"],
        "orientation_diff_snap": snap_frame["orientation_diff"],
        "orientation_diff_final": final_frame["orientation_diff"],
        "weight_diff": snap_frame["weight_diff"],
        "blocker_position": snap_frame["blocker_position"],
        "rusher_position": snap_frame["rusher_position"],
        "assisted_block": snap_frame["assisted_block"],
        "pff_sackAllowed": snap_frame["pff_sackAllowed"],
        "pff_hurryAllowed": snap_frame["pff_hurryAllowed"],
        "pff_hitAllowed": snap_frame["pff_hitAllowed"],
        "pff_beatenByDefender": snap_frame["pff_beatenByDefender"],
    })

snapshot_features = (
    paired.groupby(["gameId", "playId", "nflId", "rusher_nflId"])
    .apply(get_snapshots, include_groups=False)
    .reset_index()
)

print(f"Matchup-level rows: {len(snapshot_features):,}")

In [ ]:
outcome_cols = ["pff_sackAllowed", "pff_hurryAllowed", "pff_hitAllowed", "pff_beatenByDefender"]

snapshot_features[outcome_cols] = snapshot_features[outcome_cols].astype(bool)

snapshot_features["blocker_lost"] = (
    snapshot_features["pff_sackAllowed"]
    | snapshot_features["pff_hurryAllowed"]
    | snapshot_features["pff_hitAllowed"]
    | snapshot_features["pff_beatenByDefender"]
).astype(int)

print(snapshot_features["blocker_lost"].value_counts(normalize=True))

In [ ]:
context_cols = [
    "gameId", "playId", "down", "yardsToGo", "offenseFormation",
    "defendersInBox", "pff_playAction", "pff_passCoverage", "pff_passCoverageType"
]

play_context = plays[context_cols]
snapshot_features = snapshot_features.merge(play_context, on=["gameId", "playId"], how = "left")
print(snapshot_features[["down", "offenseFormation", "pff_passCoverage"]].isna().sum())

In [ ]:
rusher_counts = (
    pff[pff["pff_role"] == "Pass Rush"]
    .groupby(["gameId", "playId"])
    .size()
    .reset_index(name="rusher_count")
)

snapshot_features = snapshot_features.merge(rusher_counts, on = ["gameId", "playId"], how = "left")
snapshot_features["rusher_count"] = snapshot_features["rusher_count"].fillna(0).astype(int)
snapshot_features["blitz_flag"] = (snapshot_features["rusher_count"] >= 5).astype(int)

print(snapshot_features["rusher_count"].value_counts().sort_index())
print(snapshot_features["blitz_flag"].value_counts(normalize=True))

In [ ]:
for col in ["offenseFormation", "pff_passCoverage", "pff_passCoverageType"]:
    print(col, "->", snapshot_features[col].nunique(), "unique values")
    print(snapshot_features[col].value_counts(dropna=False), "\n")

In [ ]:
formation_counts = snapshot_features["offenseFormation"].value_counts(dropna=False)
thin_formations = formation_counts[formation_counts < 200].index
snapshot_features["offenseFormation"] = snapshot_features["offenseFormation"].apply(
    lambda x: "other" if pd.isna(x) or x in thin_formations else x
)

coverage_counts = snapshot_features["pff_passCoverage"].value_counts(dropna=False)
thin_coverages = coverage_counts[coverage_counts < 250].index
snapshot_features["pff_passCoverage"] = snapshot_features["pff_passCoverage"].apply(
    lambda x: "other" if pd.isna(x) or x in thin_coverages else x
)

print(snapshot_features["offenseFormation"].value_counts())
print(snapshot_features["pff_passCoverage"].value_counts())

In [ ]:
snapshot_features.to_parquet("snapshot_features.parquet", index = False)
print(f"Saved {len(snapshot_features):,} rows to snapshot_features.parquet")